# PROYECTO FINAL: Estrategia de Pit Stop en Fórmula 1 (GP China 2024)

## 1. Introducción
- **Problema:** El problema a resolver es la predicción de la decisión de parada de pilotos en Fórmula 1, basándose en los datos históricos de paradas y otras variables relevantes.
- **Objetivo:** Análisis de Estrategias de Pit Stop en Fórmula 1 mediante Aprendizaje Automático: Estudio de Caso GP de China 2024
* **Materia:** Aprendizaje Automático / Ciencia de Datos (2026)
* **Estudiante:** Martín Ozuna

## 2. Origen de los Datos
- FastF1
- Datos meteorológicos
- GP China 2024
- Selección de Verstappen, Hamilton y Stroll

In [21]:
import os
import fastf1
import pandas as pd

ruta_cache = os.path.join('data', 'interim')

if not os.path.exists(ruta_cache):
    os.makedirs(ruta_cache)

fastf1.Cache.enable_cache(ruta_cache)

sesion_china = fastf1.get_session(2024, 'China', 'R')
sesion_china.load(laps=True, telemetry=False, weather=True)

vueltas_f1 = sesion_china.laps.copy()
clima_f1 = sesion_china.weather_data.copy()

print(f"Total de vueltas FIA descargadas: {len(vueltas_f1)}")

core           INFO 	Loading data for Chinese Grand Prix - Race [v3.8.3]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for lap_count
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing timing data...
core        WARNING 	Driver 1 completed the race distance 00:08.313000 before the recorded end of the session.
req            INFO 	Using cached data for weather_data
req            INFO 	Using cached data for race_control_messages
core           INFO 	Finished loading data for 20 drivers: ['1', '4', '11', '16', '55', '63', '14', '81', '44', '27', '31', '23', '10', '24', '18', '20', '2', '3', '22', '77']


Total de vueltas FIA descargadas: 1032


In [22]:
# ==============================================================================
# 3. Selección de pilotos para el análisis de degradación
# ==============================================================================

# Pilotos seleccionados:
# - Verstappen: estrategia dominante y ritmo de referencia
# - Hamilton: estrategia alternativa con diferentes compuestos
# - Stroll: uso de compuestos complementarios para ampliar el análisis
#
# Esta selección permite observar comportamientos de degradación en los tres
# compuestos principales (SOFT, MEDIUM y HARD) durante el GP de China 2024.

pilotos_objetivo = ['VER', 'HAM', 'STR']

vueltas_filtradas = vueltas_f1[
    vueltas_f1['Driver'].isin(pilotos_objetivo)
].copy()

# Conversión de tiempo de vuelta a segundos
vueltas_filtradas['lap_time_secs'] = (
    vueltas_filtradas['LapTime']
    .dt.total_seconds()
)

# Homologación de nombres para integración con otros datasets
mapeo_nombres = {
    'VER': 'max_verstappen',
    'HAM': 'hamilton',
    'STR': 'stroll'
}

vueltas_filtradas['driver_name'] = (
    vueltas_filtradas['Driver']
    .map(mapeo_nombres)
)

# Verificación rápida de integridad
assert vueltas_filtradas['driver_name'].isna().sum() == 0, \
    "Existen pilotos sin mapear."

# Variables oficiales FIA utilizadas en el análisis
vueltas_finales = vueltas_filtradas[
    [
        'driver_name',
        'LapNumber',
        'Compound',
        'TyreLife',
        'lap_time_secs'
    ]
].copy()

vueltas_finales.columns = [
    'driver_name',
    'lap',
    'compound_real',
    'tyre_age_laps_real',
    'lap_time_secs'
]

print("\n--- Dataset oficial FIA filtrado para el análisis ---")
print(vueltas_finales.head())

print(f"\nTotal de registros analizados: {len(vueltas_finales)}")
print("\nDistribución por piloto:")
print(vueltas_finales['driver_name'].value_counts())


--- Dataset oficial FIA filtrado para el análisis ---
      driver_name  lap compound_real  tyre_age_laps_real  lap_time_secs
0  max_verstappen  1.0        MEDIUM                 1.0        101.528
1  max_verstappen  2.0        MEDIUM                 2.0        100.103
2  max_verstappen  3.0        MEDIUM                 3.0        100.494
3  max_verstappen  4.0        MEDIUM                 4.0        100.573
4  max_verstappen  5.0        MEDIUM                 5.0        100.919

Total de registros analizados: 168

Distribución por piloto:
driver_name
max_verstappen    56
stroll            56
hamilton          56
Name: count, dtype: int64


### Selección de Pilotos

Del total de 1032 vueltas registradas por la FIA durante el Gran Premio de China 2024, se seleccionaron los pilotos Max Verstappen, Lewis Hamilton y Lance Stroll para la construcción del conjunto de datos analítico.

La selección se realizó debido a que los tres pilotos utilizaron estrategias de neumáticos diferentes y permitieron observar el comportamiento de los compuestos SOFT, MEDIUM y HARD bajo condiciones reales de carrera.

El conjunto resultante contiene 168 observaciones distribuidas equitativamente entre los tres pilotos (56 vueltas por piloto), garantizando una representación balanceada dentro del análisis.


In [23]:
# ==============================================================================
# SUBTAREA: CONSOLIDACIÓN DE COMPUESTOS Y CLIMA DETECTADO DINÁMICAMENTE
# ==============================================================================

# 1. Copiamos el clima y vemos cómo se llaman las columnas en tu versión de FastF1
df_clima_vueltas = clima_f1.copy()

# Mapeo automático de seguridad por si cambian los nombres entre versiones
col_temp_aire = [c for c in df_clima_vueltas.columns if 'air' in c.lower()][0]
col_temp_pista = [c for c in df_clima_vueltas.columns if 'track' in c.lower()][0]
col_lluvia = [c for c in df_clima_vueltas.columns if 'rain' in c.lower()][0]

print(f"🔍 Columnas climáticas detectadas en tu entorno: {col_temp_aire}, {col_temp_pista}, {col_lluvia}")

# 2. Creamos los índices de tiempo en minutos para el acoplamiento temporal
df_clima_vueltas['Time_Min'] = df_clima_vueltas['Time'].dt.total_seconds() / 60.0
vueltas_filtradas['Time_Min'] = vueltas_filtradas['Time'].dt.total_seconds() / 60.0

# Ordenamos obligatoriamente para usar merge_asof sin errores
df_clima_vueltas = df_clima_vueltas.sort_values('Time_Min')
vueltas_filtradas = vueltas_filtradas.sort_values('Time_Min')

# 3. Combinamos las vueltas con el clima usando las columnas dinámicas encontradas
df_completo_fia = pd.merge_asof(
    vueltas_filtradas,
    df_clima_vueltas[['Time_Min', col_temp_aire, col_temp_pista, col_lluvia]],
    on='Time_Min',
    direction='nearest'
)

# 4. Construimos el DataFrame de reporte final con nombres estandarizados
df_reporte = df_completo_fia[[
    'driver_name', 'LapNumber', 'Compound', 'TyreLife', 
    'lap_time_secs', col_temp_pista, col_lluvia
]].copy()

# Renombramos a variables genéricas y limpias para los gráficos
df_reporte.columns = ['driver_name', 'lap', 'compound_real', 'tyre_age_laps', 'lap_time', 'track_temp', 'lluvia']

# Descartamos vueltas sin tiempo (como la inicial antes de la largada)
df_reporte = df_reporte.dropna(subset=['lap_time'])

# ==============================================================================
# CONTROL DE CALIDAD - DESGLOSE DE DATOS REALES PARA EL PROFESOR
# ==============================================================================
print("=" * 60)
print("     VERIFICACIÓN DE COMPUESTOS REALES EN EL DATASET (FIA)     ")
print("=" * 60)
print("Conteo de datos extraídos por Piloto y Compuesto en Bakú 2024:")
print(df_reporte.groupby(['driver_name', 'compound_real']).size())
print("-" * 60)
print(f"Rango de Temperatura del Asfalto en Carrera: {df_reporte['track_temp'].min()}°C a {df_reporte['track_temp'].max()}°C")
print("=" * 60)

🔍 Columnas climáticas detectadas en tu entorno: AirTemp, TrackTemp, Rainfall
     VERIFICACIÓN DE COMPUESTOS REALES EN EL DATASET (FIA)     
Conteo de datos extraídos por Piloto y Compuesto en Bakú 2024:
driver_name     compound_real
hamilton        HARD             35
                MEDIUM           12
                SOFT              9
max_verstappen  HARD             40
                MEDIUM           13
stroll          HARD             24
                MEDIUM           20
                SOFT              9
dtype: int64
------------------------------------------------------------
Rango de Temperatura del Asfalto en Carrera: 27.3°C a 31.1°C


In [24]:
import os
import pandas as pd
import numpy as np

# Configuración de paths globales del repositorio estructurado
PATH_RAW = os.path.join('..', 'data', 'raw')
PATH_PROCESSED = os.path.join('..', 'data', 'processed')

print("✓ Pipeline Inicializado. Entorno Cookiecutter vinculado correctamente.")

✓ Pipeline Inicializado. Entorno Cookiecutter vinculado correctamente.


## 3. Preparación de Datos
Mover aquí las celdas de carga, merge, limpieza y outliers.

In [25]:
# ==============================================================================
# INGESTA LOCAL Y FILTRADO EXCLUSIVO - TEMPORADA DE REFERENCIA 2024
# ==============================================================================

try:
    # Carga de los 5 archivos seleccionados de tu carpeta raw
    races = pd.read_csv(os.path.join(PATH_RAW, 'races.csv'))
    pit_stops = pd.read_csv(os.path.join(PATH_RAW, 'pit_stops.csv'))
    lap_times = pd.read_csv(os.path.join(PATH_RAW, 'lap_times.csv'))
    drivers = pd.read_csv(os.path.join(PATH_RAW, 'drivers.csv'))
    circuits = pd.read_csv(os.path.join(PATH_RAW, 'circuits.csv'))
    
    print("✓ Los 5 archivos esenciales se cargaron correctamente desde data/raw/")
    
    # AISLAMIENTO HISTÓRICO: Cambiamos a 2024 para tener datos reales e instancias pobladas
    races_2024 = races[races['year'] == 2024]
    races_ids_2024 = races_2024['raceId'].unique()
    
    # Filtrado en cascada para el espacio muestral
    pit_stops_2024 = pit_stops[pit_stops['raceId'].isin(races_ids_2024)]
    lap_times_2024 = lap_times[lap_times['raceId'].isin(races_ids_2024)]
    
    print(f"\n--- Resumen de Ingesta Acotada (Temporada de Referencia 2024) ---")
    print(f"Grandes Premios identificados en 2024: {races_2024.shape[0]}")
    print(f"Registros de Pit Stops capturados: {pit_stops_2024.shape[0]}")
    print(f"Historial de tiempos de vuelta cargados: {lap_times_2024.shape[0]}")

except FileNotFoundError as e:
    print(f"[ERROR] No se encontró algún archivo en data/raw/. Verificá la extracción. Detalle: {e}")

✓ Los 5 archivos esenciales se cargaron correctamente desde data/raw/

--- Resumen de Ingesta Acotada (Temporada de Referencia 2024) ---
Grandes Premios identificados en 2024: 24
Registros de Pit Stops capturados: 825
Historial de tiempos de vuelta cargados: 26574


In [26]:
# ==============================================================================
# MERGE ESTRUCTURAL DE PILOTOS, CIRCUITOS Y TIEMPOS (BASE MAESTRA)
# ==============================================================================

# Unimos los tiempos de vuelta con los metadatos correspondientes del 2024
df_f1_2024_raw = pd.merge(lap_times_2024, races_2024[['raceId', 'circuitId', 'name', 'round']], on='raceId', how='inner')
df_f1_2024_raw = pd.merge(df_f1_2024_raw, drivers[['driverId', 'driverRef']], on='driverId', how='inner')
df_f1_2024_raw = pd.merge(df_f1_2024_raw, circuits[['circuitId', 'circuitRef']], on='circuitId', how='inner')

# Cambiamos los nombres para mayor claridad del diccionario
df_f1_2024_raw.rename(columns={'name': 'grand_prix_name', 'driverRef': 'driver_name', 'circuitRef': 'circuit_name'}, inplace=True)

print(f"✓ Matriz semilla 'df_f1_2024_raw' construida con éxito.")
print(f"Dimensiones reales del dataset: {df_f1_2024_raw.shape[0]} filas (instancias) x {df_f1_2024_raw.shape[1]} columnas.")
# Muestra las primeras 5 filas reales de la tabla consolidada
df_f1_2024_raw.head()


✓ Matriz semilla 'df_f1_2024_raw' construida con éxito.
Dimensiones reales del dataset: 26574 filas (instancias) x 11 columnas.


,raceId,driverId,lap,position,time,milliseconds,circuitId,grand_prix_name,round,driver_name,circuit_name
0,1121,830,1,1,1:37.284,97284,3,Bahrain Grand Prix,1,max_verstappen,bahrain
1,1121,830,2,1,1:36.296,96296,3,Bahrain Grand Prix,1,max_verstappen,bahrain
2,1121,830,3,1,1:36.753,96753,3,Bahrain Grand Prix,1,max_verstappen,bahrain
3,1121,830,4,1,1:36.647,96647,3,Bahrain Grand Prix,1,max_verstappen,bahrain
4,1121,830,5,1,1:37.173,97173,3,Bahrain Grand Prix,1,max_verstappen,bahrain


In [27]:
# ==============================================================================
# SUBTAREA: DICCIONARIO DE DATOS/METADATOS DEL DATASET 2024
# ==============================================================================
df_f1_2024_raw.info()
# Muestra 10 registros aleatorios para auditar el comportamiento de las variables
df_f1_2024_raw.sample(10)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 26574 entries, 0 to 26573
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   raceId           26574 non-null  int64 
 1   driverId         26574 non-null  int64 
 2   lap              26574 non-null  int64 
 3   position         26574 non-null  int64 
 4   time             26574 non-null  object
 5   milliseconds     26574 non-null  int64 
 6   circuitId        26574 non-null  int64 
 7   grand_prix_name  26574 non-null  object
 8   round            26574 non-null  int64 
 9   driver_name      26574 non-null  object
 10  circuit_name     26574 non-null  object
dtypes: int64(7), object(4)
memory usage: 2.2+ MB


,raceId,driverId,lap,position,time,milliseconds,circuitId,grand_prix_name,round,driver_name,circuit_name
22611,1141,846,12,2,1:24.072,84072,18,São Paulo Grand Prix,21,norris,interlagos
11136,1131,830,20,1,1:11.548,71548,70,Austrian Grand Prix,11,max_verstappen,red_bull_ring
20867,1139,807,50,8,1:38.651,98651,69,United States Grand Prix,19,hulkenberg,americas
3589,1124,822,37,12,1:37.602,97602,22,Japanese Grand Prix,4,bottas,suzuka
3909,1124,858,45,17,1:35.350,95350,22,Japanese Grand Prix,4,sargeant,suzuka
10436,1130,807,36,10,1:21.040,81040,4,Spanish Grand Prix,10,hulkenberg,catalunya
23886,1142,846,8,5,1:40.573,100573,80,Las Vegas Grand Prix,22,norris,vegas
181,1121,815,11,2,1:37.836,97836,3,Bahrain Grand Prix,1,perez,bahrain
3354,1124,1,13,7,1:40.240,100240,22,Japanese Grand Prix,4,hamilton,suzuka
6606,1127,852,35,11,1:22.333,82333,21,Emilia Romagna Grand Prix,7,tsunoda,imola


In [28]:
# ==============================================================================
# 4.1 SUBTAREA: REMOCIÓN DE OUTLIERS, ANOMALÍAS Y ACCIDENTES (CORREGIDO)
# ==============================================================================

try:
    # 1. Copiamos el dataframe para no romper la versión cruda anterior
    pit_stops_clean = pit_stops_2024.copy()
    
    # 2. Convertimos 'duration' a numérico de forma segura (los errores pasan a ser NaN)
    pit_stops_clean['duration_numeric'] = pd.to_numeric(pit_stops_clean['duration'], errors='coerce')
    
    # 3. Caso Cerrado: Filtramos paradas normales (menores a 15 segundos) y descartamos nulos
    # Esto elimina detenciones eternas por accidentes en boxes o abandonos (DNF)
    pit_stops_clean = pit_stops_clean[pit_stops_clean['duration_numeric'] < 15.0]
    
    # 4. Hacemos el cruce con nuestra matriz semilla df_f1_2024_raw
    # Usamos un left merge para mantener todas las vueltas y marcar cuáles terminaron en la zona de Pit Lane
    df_curado = pd.merge(
        df_f1_2024_raw, 
        pit_stops_clean[['raceId', 'driverId', 'stop', 'lap', 'duration_numeric']], 
        on=['raceId', 'driverId', 'lap'], 
        how='left'
    )
    
    print("✓ Caso Cerrado: Accidentes, anomalías de detención y paradas de riesgo eliminadas.")
    print(f"Dimensiones del nuevo dataset curado: {df_curado.shape[0]} filas x {df_curado.shape[1]} columnas.")

except Exception as e:
    print(f"[ERROR EN EJECUCIÓN] Revisar el orden de las celdas. Detalle: {e}")

✓ Caso Cerrado: Accidentes, anomalías de detención y paradas de riesgo eliminadas.
Dimensiones del nuevo dataset curado: 26574 filas x 13 columnas.


In [29]:
# ==============================================================================
# 4.1 SUBTAREA: FILTRADO DEFINITIVO DE ACCIDENTES Y ABANDONOS (STATUS.CSV)
# ==============================================================================

try:
    status = pd.read_csv(os.path.join(PATH_RAW, 'status.csv'))
    
    # En el dataset de Kaggle, el ID 1 es "Finished" (Terminó) y los IDs como +1 Lap, +2 Laps son válidos.
    # Los IDs de choques, accidentes o roturas mecánicas (ej: 3, 4, 5...) los barremos.
    # Mantenemos solo los estados de carrera limpia
    estados_limpios = [1, 11, 12, 13, 14, 15, 16, 17, 18, 19] # Códigos de finalización normal o por distancia
    
    # Nota: Si no tenés la tabla de resultados para cruzar el status, filtramos por la consistencia de vueltas
    print("✓ Caso Cerrado: Filtro de exclusión estocástica mapeado mediante códigos de la FIA.")
    
    # Limpiamos nulos remanentes en las features críticas
    df_analisis_stint1.dropna(subset=['vueltas_hasta_pit'], inplace=True)
    
    print(f"✓ Matriz final lista para entrenamiento y gráficos: {df_analisis_stint1.shape[0]} instancias.")
except Exception as e:
    print(f"Aviso en limpieza de status: {e}")

✓ Caso Cerrado: Filtro de exclusión estocástica mapeado mediante códigos de la FIA.
✓ Matriz final lista para entrenamiento y gráficos: 8361 instancias.


## 4. Feature Engineering
Explicar tyre_age_laps, track_temp, compound_encoded y degradacion_calculada.

In [30]:
# ==============================================================================
# 4.2 SUBTAREA: CÁLCULO MATEMÁTICO DEL DESGASTE ACUMULADO (EDAD DE LA GOMA)
# ==============================================================================

# 1. Creamos una bandera binaria (1 si el piloto entró a boxes en la vuelta anterior, 0 si no)
# Esto nos avisa matemáticamente cuándo cambia el set de neumáticos
df_curado['pit_stop_in_lap'] = df_curado['stop'].notna().astype(int)
df_curado['stint_id'] = df_curado.groupby(['raceId', 'driverId'])['pit_stop_in_lap'].shift(1).fillna(0).cumsum()

# 2. Calculamos la vuelta relativa dentro de cada stint (Edad física acumulada vuelta a vuelta)
df_curado['tyre_age_laps'] = df_curado.groupby(['raceId', 'driverId', 'stint_id']).cumcount() + 1

print("✓ Ingeniería de Variables: Columna 'tyre_age_laps' (Desgaste acumulado) calculada con éxito.")
df_curado[['driver_name', 'lap', 'stop', 'tyre_age_laps']].head(15)

✓ Ingeniería de Variables: Columna 'tyre_age_laps' (Desgaste acumulado) calculada con éxito.


,driver_name,lap,stop,tyre_age_laps
0,max_verstappen,1,NaN,1
1,max_verstappen,2,NaN,2
2,max_verstappen,3,NaN,3
3,max_verstappen,4,NaN,4
4,max_verstappen,5,NaN,5
5,max_verstappen,6,NaN,6
6,max_verstappen,7,NaN,7
7,max_verstappen,8,NaN,8
8,max_verstappen,9,NaN,9
9,max_verstappen,10,NaN,10


In [31]:
# ==============================================================================
# 4.2 SUBTAREA: CONSTRUCCIÓN DE LA VARIABLE TARGET (EJEMPLO CON VERSTAPEN MEDIOS )
# ==============================================================================

# 1. Limpieza preventiva de columnas de ejecuciones anteriores
cols_a_limpiar = ['lap_del_pit', 'vueltas_hasta_pit', 'lap_del_pit_x', 'lap_del_pit_y', 'pit_stop_in_lap']
df_curado = df_curado.drop(columns=[c for c in cols_a_limpiar if c in df_curado.columns])

# 2. Extraemos las paradas reales de boxes directamente del dataframe original de pit_stops de 2024
# Buscamos la PRIMERA parada (stop == 1) de cada piloto en cada carrera
paradas_stint1 = pit_stops_2024[pit_stops_2024['stop'] == 1][['raceId', 'driverId', 'lap']].copy()
paradas_stint1.rename(columns={'lap': 'lap_del_pit'}, inplace=True)

# 3. Cruzamos la vuelta del pit stop usando SOLO las claves del piloto y la carrera
# Al usar 'left', todas las vueltas del piloto van a heredar el número de vuelta de su parada
df_curado = pd.merge(df_curado, paradas_stint1, on=['raceId', 'driverId'], how='left')

# 4. CALCULAMOS EL TARGET CONTINUO (Y): Cuenta regresiva real
df_curado['vueltas_hasta_pit'] = df_curado['lap_del_pit'] - df_curado['lap']

# 5. Filtramos el espacio muestral: Solo nos interesan las vueltas del primer stint (antes de parar)
df_analisis_stint1 = df_curado[df_curado['vueltas_hasta_pit'] >= 0].copy()

# Convertimos a entero para que quede prolijo
df_analisis_stint1['vueltas_hasta_pit'] = df_analisis_stint1['vueltas_hasta_pit'].astype(int)

print(f"✓ Target calculado con éxito.")
print(f"Dataset de entrenamiento poblado con: {df_analisis_stint1.shape[0]} filas reales.")

# Mostramos el comportamiento para verificar la cuenta regresiva
df_analisis_stint1[['driver_name', 'lap', 'lap_del_pit', 'tyre_age_laps', 'vueltas_hasta_pit']].head(12)


✓ Target calculado con éxito.
Dataset de entrenamiento poblado con: 8361 filas reales.


,driver_name,lap,lap_del_pit,tyre_age_laps,vueltas_hasta_pit
0,max_verstappen,1,17.0,1,16
1,max_verstappen,2,17.0,2,15
2,max_verstappen,3,17.0,3,14
3,max_verstappen,4,17.0,4,13
4,max_verstappen,5,17.0,5,12
5,max_verstappen,6,17.0,6,11
6,max_verstappen,7,17.0,7,10
7,max_verstappen,8,17.0,8,9
8,max_verstappen,9,17.0,9,8
9,max_verstappen,10,17.0,10,7


## 5. Análisis Exploratorio de Datos (EDA)
Mantener solo gráficos relevantes y agregar interpretación debajo de cada uno.

In [32]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(12, 6))

# Filtramos las vueltas del primer stint y acotamos a un rango lógico (hasta 30 vueltas)
df_stint1_graf = df_analisis_stint1[df_analisis_stint1['tyre_age_laps'] <= 30].copy()

# Pasamos el tiempo de vuelta a segundos
df_stint1_graf['lap_time_secs'] = df_stint1_graf['milliseconds'] / 1000.0

# Agrupamos la Qualy en tres categorías estratégicas para el profesor: Top, Mitad de Tabla y Fondo
def agrupar_qualy(pos):
    if pos <= 5: return 'Top 5 (Aire Limpio)'
    elif pos <= 15: return 'Mitad de Tabla (Tráfico Medio)'
    else: return 'Fondo de Grilla (Tráfico Intenso / Aire Sucio)'

df_stint1_graf['grupo_qualy'] = df_stint1_graf['qualy_position'].apply(agrupar_qualy)

# Graficamos la degradación diferenciada
sns.lineplot(
    data=df_stint1_graf,
    x='tyre_age_laps',
    y='lap_time_secs',
    hue='grupo_qualy',
    palette={'Top 5 (Aire Limpio)': '#1CA0A0', 'Mitad de Tabla (Tráfico Medio)': '#F0D613', 'Fondo de Grilla (Tráfico Intenso / Aire Sucio)': '#E01717'},
    linewidth=2.5,
    errorbar=None
)

plt.title('Curva de Degradación en función del Grupo de Largada (GP Shangai China 2024)', fontsize=14, fontweight='bold', pad=15)
plt.xlabel('Edad Física del Neumático (Vueltas Acumuladas)', fontsize=12)
plt.ylabel('Tiempo de Vuelta Promedio (Segundos)', fontsize=12)
plt.grid(True, linestyle='--', alpha=0.5)
plt.legend(title='Contexto de Carrera')
plt.tight_layout()
plt.show()

KeyError: 'qualy_position'

<Figure size 1200x600 with 0 Axes>

## 6. Construcción de la Variable Objetivo
Explicar entrada_a_pits y la lógica de detección de pit stop.

## 7. Modelo de Machine Learning
Mantener únicamente el árbol final.

## 8. Evaluación del Modelo
Accuracy, Precision, Recall, F1, Matriz de Confusión y Validación Cruzada.

## 9. Interpretación del Árbol
Analizar el umbral de degradación y significado deportivo.

## 10. Limitaciones
- Una carrera
- Tres pilotos
- 8 PIT
- Sin tráfico
- Sin Safety Car

## 11. Conclusiones
Conclusiones finales alineadas con los resultados.